# Phytoplankton OSR — Unknown Discovery via OpenMax-Gated HDBSCAN Clustering (Colab)

This notebook runs the **clustering stage** of the pipeline:





1.   load a trained SYKE-PIC model,
2.   compute OpenMax unknownness on OSR test images,
3.  select the top-q most “unknown” samples,
4. cluster their embeddings with HDBSCAN,
5. export CSV summaries + UMAP visualizations and cluster montages.

# 0) Prerequisites

You must already have:



*   **Split folders** in Colab local disk at:
/content/data/osr_test_root/
(and optionally train/val roots, but clustering uses only osr_test_root)
*   A trained model directory on Google Drive at:
/content/drive/MyDrive/SykePic/Outputs/Final/resnet18_1
containing at least:  
```
# 1. class_names.txt
# 2. openmax_mavs_logits_best.npy
# 3. openmax_weibull_logits_best.pkl
# 4. the trained weights (best.pth / equivalent required by sykepic.prepare_model)

```

# 1) Clone repo + install

In [1]:
!git clone https://github.com/ian-of-yore/phytoplankton-osr.git
%cd phytoplankton-osr
!pip install -e .

Cloning into 'phytoplankton-osr'...
remote: Enumerating objects: 100, done.
remote: Counting objects: 100% (100/100), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 100 (delta 39), reused 85 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (100/100), 42.85 KiB | 2.38 MiB/s, done.
Resolving deltas: 100% (39/39), done.
/content/phytoplankton-osr
Obtaining file:///content/phytoplankton-osr
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for phytoplankton-osr (pyproject.toml) ... done
  Created wheel for phytoplankton-osr: filename=phytoplankton_osr-0.1.0-0.editable-py3-none-any.whl size=1294 sha256=7cafc83a98c9b9cfef5c9c67a040ad8efdc76c9555e17f6d2ba10322fb8632bf
  Stored in directory: /tmp/pip-ephem-wheel-cache-6qd2t2sg/wheels/ff/22/4d/687c7e3dcc8fe99aa0455952951d2c5267925441d8fd3

# 2) Clone SYKE-PIC + install feature dependency

In [2]:
%%bash
set -e
cd /content

rm -rf syke-pic
git clone https://github.com/sykefi/syke-pic.git
cd syke-pic

python -m pip install --quiet "numpy>=1.26,<2.2.0" "git+https://github.com/veot/ifcb-features"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 47.8 MB/s eta 0:00:00


Cloning into 'syke-pic'...


# 3) Install clustering dependencies

In [3]:
!pip install -q umap-learn hdbscan

# 4) Import the datapath and trained model directory

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!unzip "/content/drive/MyDrive/SykePic/Data/iowFinalMinCount50.zip" -d "/content/data/"

Streaming output truncated to the last 5000 lines.
  inflating: /content/data/osr_test_root/Aphanizomenon_bundle/D20240730T100623_IFCB197_00046.png  
  inflating: /content/data/osr_test_root/Aphanizomenon_bundle/D20240604T104337_IFCB197_00004.png  
  inflating: /content/data/osr_test_root/Aphanizomenon_bundle/D20240730T100623_IFCB197_01363.png  
  inflating: /content/data/osr_test_root/Aphanizomenon_bundle/D20240730T100623_IFCB197_00649.png  
  inflating: /content/data/osr_test_root/Aphanizomenon_bundle/D20240730T100623_IFCB197_00220.png  
  inflating: /content/data/osr_test_root/Aphanizomenon_bundle/D20240730T111700_IFCB197_00474.png  
  inflating: /content/data/osr_test_root/Aphanizomenon_bundle/D20240730T111700_IFCB197_00072.png  
  inflating: /content/data/osr_test_root/Aphanizomenon_bundle/D20240730T100623_IFCB197_01495.png  
  inflating: /content/data/osr_test_root/Centrales_single_cells/D20240312T082736_IFCB197_00461.png  
  inflating: /content/data/osr_test_root/Centrales_singl

# 5) Create Colab paths config

In [6]:
%%bash
cat > /content/paths_colab.yaml <<'YAML'
paths:
  out_base: "/content/data"  # Where the four splited dataset remains. Out of those four, "osr_test_root" will be used in clustering
  model_out_dir: "/content/drive/MyDrive/SykePic/Outputs/Final"
  model_dir: "/content/drive/MyDrive/SykePic/Outputs/Final/resnet18_1"
  sykepic_repo: "/content/syke-pic"
YAML

# 5) Experiment Configuration

The main experiment configuration is stored in:

`configs/exp.yaml`  (edit this file in the repository)

Below are the key parameters relevant for clustering.

---

## OpenMax (Unknownness Gate)

**openmax.alpha**  
Controls how many of the top-logit classes are recalibrated by OpenMax.  
Recommended: `alpha: 10` (this produced the best results in our experiments).

**openmax.tailsize**  
Tail size used during Weibull fitting for OpenMax calibration.  
(This is already determined during the OpenMax fitting stage.)

Note: Clustering uses the OpenMax artifacts saved inside `model_dir/`
(`openmax_mavs_logits_*.npy`, `openmax_weibull_logits_*.pkl`)
and applies the stored parameters when computing `P_unknown`.

---

## Clustering (HDBSCAN on Embeddings)

**clustering.q_unknown**  
Fraction of OSR test samples selected as unknown candidates based on OpenMax `P_unknown`.  
Example: `0.25` selects the top 25% most-unknown samples.

**clustering.min_cluster_size**  
Minimum size required for a cluster in HDBSCAN.  
Example: `5` allows smaller clusters.  
Increasing this enforces larger, more stable clusters.

**clustering.min_samples**  
Density strictness parameter in HDBSCAN.  
Higher values → more conservative clustering → more samples labeled as noise (-1).

**clustering.umap_neighbors**, **clustering.umap_min_dist**  
UMAP visualization parameters only.  
These do NOT affect clustering labels — only the 2D projection.

**clustering.out_prefix**  
Prefix used for saved output files:
`MODEL_DIR/clusters/<out_prefix>_*`

**clustering.seed**  
Random seed used for UMAP projection.

---

## What You Typically Change

In `configs/exp.yaml`, you usually modify:

- `openmax.alpha`
- `clustering.q_unknown`
- `clustering.min_cluster_size`
- `clustering.min_samples`
- `clustering.out_prefix`

---



# 6) Recommended “working” settings
The configuration that gave stable, expected results:



```
openmax:
  alpha: 10

clustering:
  q_unknown: 0.25
  min_cluster_size: 5
  min_samples: 3
  umap_neighbors: 20
  umap_min_dist: 0.1
  out_prefix: "discover_openmax_q0.25_mcs5_ms3"
  seed: 42
```
(Apply these values inside **/content/phytoplankton-osr/configs/exp.yaml**.)



# 7) Run clustering

In [7]:
!python scripts/run_clustering_colab.py --paths /content/paths_colab.yaml --exp configs/exp.yaml

2026-03-04 16:20:37.152610: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-04 16:20:37.611398: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-04 16:20:37.884769: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772641238.199634    6011 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772641238.287105    6011 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772641238.887373    6011 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin